# NIH ChestX-ray14: DVC stage `preprocess` on Colab

**For education only, not for clinical use.**

Downloads the NIH dataset from Kaggle (`nih-chest-xrays/data`) to the Colab VM and runs the DVC
stage `preprocess`: every image in `data/splits` is resized to 224x224 8-bit grayscale and stored
in zip shards with a sha256 manifest. The ImageNet mean and std are stage parameters, written to
`preprocess.json`, and applied by the training loader.

**Storage:** everything is written to the Colab VM disk. Drive receives only `dvc push` (the DVC
remote `MyDrive/aiml02-dvc`) and a copy of `dvc.lock`. Nothing on Drive is deleted.

- **Not run yet.** The code it calls (`src/data/preprocess.py`) is tested on synthetic images only.
- Use a **CPU runtime**: no GPU is needed, and CPU runtimes have more disk.
- **Disk is checked first** (`df -h`), and the notebook stops if the VM or Drive is short of space.
- **The zip is never extracted.** Images are read straight from it; only the three small metadata
  files are copied out.
- **Resumable within a session:** shards are built in `/content/work/nih_224` (VM) and recorded in
  its manifest only when complete, so rerunning section 7 after an error skips finished shards.
  After a disconnect the VM disk is wiped and the run starts over.
- **The zip is deleted only when you confirm it**, in the last section, and only after the
  outputs and the manifest are complete and verified.
- Kaggle credentials are optional for this public dataset. If Kaggle asks for them, add
  `KAGGLE_USERNAME` and `KAGGLE_KEY` as Colab secrets.

## 1. Disk space first

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def run(*args, cwd=None, env=None):
    """Run a command, stream its output into the notebook, fail on a non-zero exit."""
    process = subprocess.Popen(
        [str(arg) for arg in args], cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True,
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"{args[0]} failed with exit code {process.returncode}")


def free_gb(path):
    return shutil.disk_usage(path).free / 1e9


run("df", "-h", "/content")

RAW = Path("/content/raw")  # the Kaggle zip, on the VM
RESUME = Path("/content/work/nih_224")  # shards being built, on the VM
for path in (RAW, RESUME):
    assert not str(path.resolve()).startswith("/content/drive"), f"{path} must be on the VM disk"
    path.mkdir(parents=True, exist_ok=True)
ZIP_GB = 45  # Kaggle zip of nih-chest-xrays/data
# 224px shards (estimate, kept high). Two copies on the VM: the resume folder and the stage output.
# The DVC cache adds none, because it is set to hardlink below.
OUTPUT_GB = 6
MARGIN_GB = 5

have_zip = any(RAW.glob("*.zip"))
need_vm = (0 if have_zip else ZIP_GB) + 2 * OUTPUT_GB + MARGIN_GB
print(f"VM: {free_gb('/content'):.1f} GB free, {need_vm} GB needed (zip already here: {have_zip})")
if free_gb("/content") < need_vm:
    raise RuntimeError("Not enough VM disk. Stop here: use a CPU runtime or free some space.")

## 2. Drive: room for the DVC remote

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DVC_REMOTE = Path("/content/drive/MyDrive/aiml02-dvc")
RUNS = Path("/content/drive/MyDrive/aiml02-runs/preprocess")
for path in (DVC_REMOTE, RUNS):
    path.mkdir(parents=True, exist_ok=True)

run("df", "-h", "/content/drive")
# Only the DVC remote holds the shards on Drive; files already pushed are not sent again.
in_remote_gb = sum(path.stat().st_size for path in DVC_REMOTE.rglob("*") if path.is_file()) / 1e9
need_drive = max(0.0, OUTPUT_GB - in_remote_gb) + 1
print(f"Drive: {free_gb('/content/drive'):.1f} GB free, {need_drive:.1f} GB needed "
      f"({in_remote_gb:.2f} GB already in the DVC remote)")
if free_gb("/content/drive") < need_drive:
    raise RuntimeError("Not enough Drive space. Stop here and free some space on Drive.")

## 3. Code from the repo, and DVC

In [ ]:
REPO_URL = "https://github.com/manojkumargudge/medical-imaging-ai.git"
REPO_REF = "main"
REPO = Path("/content/medical-imaging-ai")

if not REPO.exists():
    run("git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO)
commit = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print("repo commit", commit)
# On the path rather than installed: the repo pins are for local Python 3.11, and Colab's own
# packages (Pillow, PyYAML, pandas, torch with CUDA) are used here.
sys.path.insert(0, str(REPO))
os.chdir(REPO)

run(sys.executable, "-m", "pip", "install", "-q", "dvc==3.67.1")
remote = subprocess.run(
    ["dvc", "remote", "list"], cwd=REPO, capture_output=True, text=True, check=True
).stdout
print(remote)
assert "/content/drive/MyDrive/aiml02-dvc" in remote, "DVC remote is not the Drive folder"
# The cache hardlinks to the workspace instead of holding a second copy on the VM disk.
run("dvc", "config", "--local", "cache.type", "hardlink,copy", cwd=REPO)

from src.data import preprocess

## 4. Kaggle CLI (credentials optional)

In [ ]:
run(sys.executable, "-m", "pip", "install", "-q", "kaggle==2.2.5", "six==1.17.0")

try:
    from google.colab import userdata

    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Using Kaggle credentials from Colab secrets")
except Exception as exc:
    print(f"No Kaggle secrets ({type(exc).__name__}); trying without credentials")

SLUG = "nih-chest-xrays/data"
listing = subprocess.run(
    ["kaggle", "datasets", "files", SLUG, "--page-size", "20"],
    capture_output=True,
    text=True,
    check=True,
)
print(listing.stdout)
assert "Data_Entry_2017.csv" in listing.stdout, f"Unexpected listing for {SLUG}: {listing.stderr}"

## 5. Download to the VM (about 45 GB, not extracted)

In [ ]:
if not any(RAW.glob("*.zip")):
    run("kaggle", "datasets", "download", SLUG, "-p", RAW)

zips = sorted(RAW.glob("*.zip"))
assert len(zips) == 1, f"Expected one zip, found {zips}"
ZIP = zips[0]
print(ZIP, f"{ZIP.stat().st_size / 1e9:.1f} GB")
run("df", "-h", "/content")

## 6. Image count, and the metadata files for the `split` stage

In [ ]:
import hashlib
import zipfile

import yaml

EXPECTED_IMAGES = 112_120

images = preprocess.ZipImages(ZIP)
names = images.names
images.close()
print(f"PNG images in zip: {len(names)}")
assert len(names) == EXPECTED_IMAGES, f"Expected {EXPECTED_IMAGES} images, found {len(names)}"

# Only these three small files are copied out of the zip. They must match dvc.lock, or the
# split (and every result after it) would change.
lock = yaml.safe_load((REPO / "dvc.lock").read_text())
expected = {
    Path(dep["path"]).name: dep["md5"]
    for dep in lock["stages"]["split"]["deps"]
    if dep["path"].startswith("data/raw/")
}
raw_dir = REPO / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP) as archive:
    members = {Path(name).name: name for name in archive.namelist() if Path(name).name in expected}
    assert set(members) == set(expected), f"zip lacks {set(expected) - set(members)}"
    for name, member in members.items():
        target = raw_dir / name
        if not target.exists():
            target.write_bytes(archive.read(member))
        md5 = hashlib.md5(target.read_bytes()).hexdigest()
        print(name, md5, "matches dvc.lock" if md5 == expected[name] else "DIFFERS from dvc.lock")
        if md5 != expected[name]:
            raise RuntimeError(f"{name} differs from dvc.lock; stop and compare before going on")

## 7. Run the DVC stage `preprocess`

`dvc repro` first rebuilds `data/splits` (stage `split`, deterministic: same md5 as `dvc.lock`),
then resizes about 112k images into about 56 shards. Shards finished in an earlier session are
checked and skipped. A shard in the resume folder that differs from what this run would write is
never overwritten: the run stops.

In [ ]:
env = {**os.environ, "NIH_ZIP": str(ZIP), "AIML02_RESUME_DIR": str(RESUME)}
run("dvc", "repro", "preprocess", cwd=REPO, env=env)
run("df", "-h", "/content", "/content/drive")

## 8. Verify the outputs and the manifest

In [ ]:
import csv
import json

OUT = REPO / "data" / "processed" / "nih_224"
MANIFEST = REPO / "data" / "manifests" / "nih_224.csv"

verified = preprocess.verify_shards(OUT)
with MANIFEST.open(newline="") as file:
    rows = list(csv.DictReader(file))
info = json.loads((OUT / preprocess.PREPROCESS_INFO).read_text())
print(f"zip images {len(names)}, shard images {verified.images}, manifest rows {len(rows)}")
assert len(names) > 0 and verified.images > 0 and len(rows) > 0
assert verified.images == len(names) == len(rows), "outputs are incomplete"

patients = {}
for row in rows:
    patients.setdefault(row["split"], set()).add(row["patient"])
for split, members in sorted(patients.items()):
    print(f"{split}: {sum(row['split'] == split for row in rows)} images, {len(members)} patients")
assert set(patients) == {"train", "val", "test"} and all(patients.values())
splits = sorted(patients)
for i, first in enumerate(splits):
    for second in splits[i + 1:]:
        overlap = patients[first] & patients[second]
        print(f"patient overlap {first}/{second}: {len(overlap)}")
        assert not overlap
print(json.dumps(info, indent=2))
print(f"{verified.shards} shards, {verified.total_bytes / 1e9:.2f} GB")
MANIFEST_VERIFIED = True

## 9. Push to the DVC remote on Drive

`dvc.lock` now has the `preprocess` entry. It is copied to `MyDrive/aiml02-runs/preprocess/` so it
can be committed from the laptop (see the README, "Data versioning").

In [ ]:
run("dvc", "push", cwd=REPO)
status = subprocess.run(
    ["dvc", "status", "--cloud"], cwd=REPO, capture_output=True, text=True, check=True
).stdout
print(status)
assert "are in sync" in status, "the remote is missing some files"

lock_copy = RUNS / f"dvc.lock.{commit[:7]}"
if lock_copy.exists() and lock_copy.read_text() != (REPO / "dvc.lock").read_text():
    raise RuntimeError(f"{lock_copy} exists with other content; not overwriting")
shutil.copyfile(REPO / "dvc.lock", lock_copy)
print("copied dvc.lock to", lock_copy)
print((REPO / "dvc.lock").read_text())

## 10. Delete the zip from the VM (asks first)

Only the VM copy of the Kaggle zip is deleted; it can be downloaded again. This cell refuses to
run unless section 8 passed, and deletes only if you type `DELETE`.

In [ ]:
if not globals().get("MANIFEST_VERIFIED"):
    raise RuntimeError("Run section 8 first: the manifest is not verified")
assert verified.images == len(rows) == len(names) > 0

size_gb = ZIP.stat().st_size / 1e9
answer = input(f"Delete {ZIP} ({size_gb:.1f} GB) from the VM? Type DELETE to confirm: ")
if answer.strip() == "DELETE":
    ZIP.unlink()
    print("deleted", ZIP)
else:
    print("kept", ZIP)
run("df", "-h", "/content")

## 11. Spot check

In [ ]:
import io

import matplotlib.pyplot as plt
from PIL import Image

with zipfile.ZipFile(OUT / "shard-00000.zip") as archive:
    first = archive.namelist()[0]
    image = Image.open(io.BytesIO(archive.read(first)))
    print(first, image.mode, image.size)
    plt.imshow(image, cmap="gray")
    plt.title(first)
    plt.axis("off")
    plt.show()